# CoA Instrument Identity Probe

**Purpose:** Explore whether NCIt structural classes C20993 (instruments) and C211913 (question containers)
can provide machine-actionable instrument identity for the 359 SDTM CT instrument codelists.

**Inputs:**
- `sdtm-test-codes/downloads/Thesaurus.txt` — NCIt FLAT file (cached by green track)
- `sdtm-test-codes/interim/SDTM_CT_Instrument_Extract.csv` — Green track instrument extract

**Key questions:**
1. What does the C20993 hierarchy actually look like? (Spoiler: deeper than expected)
2. Can we structurally map C211913 question containers to green codelists?
3. Can we match green codelists to C20993 instrument identity?
4. What is the unresolved remainder and what patterns does it show?

*cdisc-for-ai, explore/coa-structure branch, April 2026*


In [ ]:
import csv
import os
from collections import defaultdict, Counter
import pandas as pd

# Paths relative to repo root — notebook is in coa-structure/notebooks/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
FLAT_PATH = os.path.join(REPO_ROOT, 'sdtm-test-codes', 'downloads', 'Thesaurus.txt')
GREEN_INSTRUMENT_PATH = os.path.join(REPO_ROOT, 'sdtm-test-codes', 'interim', 'SDTM_CT_Instrument_Extract.csv')
INTERIM_PATH = os.path.join(REPO_ROOT, 'coa-structure', 'interim')

assert os.path.exists(FLAT_PATH), f'Thesaurus FLAT not found at {FLAT_PATH}'
assert os.path.exists(GREEN_INSTRUMENT_PATH), f'Green instrument extract not found at {GREEN_INSTRUMENT_PATH}'
print(f'Repo root: {REPO_ROOT}')
print(f'FLAT file: {os.path.basename(FLAT_PATH)}')
print(f'Green extract: {os.path.basename(GREEN_INSTRUMENT_PATH)}')

## 1. Parse NCIt FLAT file

Tab-separated: `code \t uri \t parent_code \t names(pipe-separated) \t definition \t ... \t semantic_type \t codelist_membership`

We need: code, parent, preferred term, synonyms, definition.


In [ ]:
parent_map = {}   # code -> parent_code
name_map = {}     # code -> preferred_term
syn_map = {}      # code -> [synonyms]
defn_map = {}     # code -> definition

with open(FLAT_PATH, 'r', encoding='utf-8') as f:
    reader = csv.reader(f, delimiter='\t')
    for row in reader:
        if len(row) < 4:
            continue
        code = row[0]
        parent_map[code] = row[2]
        names = row[3].split('|')
        name_map[code] = names[0].strip()
        syn_map[code] = [n.strip() for n in names[1:] if n.strip()]
        defn_map[code] = row[4].strip() if len(row) > 4 else ''

print(f'Total NCIt concepts loaded: {len(parent_map):,}')


In [ ]:
# Build children_of index once
children_of = defaultdict(set)
for code, parent in parent_map.items():
    children_of[parent].add(code)

def get_descendants(root):
    """Transitive walk: all descendants of root."""
    descendants = set()
    queue = list(children_of[root])
    while queue:
        c = queue.pop()
        if c not in descendants:
            descendants.add(c)
            queue.extend(children_of[c])
    return descendants

def get_depth(code, root):
    """Depth of code relative to root (root's children = depth 1)."""
    d = 0
    c = code
    while c != root and d < 10:
        c = parent_map.get(c, '')
        d += 1
    return d if c == root else -1

print('Helper functions ready.')


## 2. C20993 — Research or Clinical Assessment Tool

The instrument identity layer. The plan assumed 274 flat descendants.
Let's see what the hierarchy actually looks like.


In [ ]:
c20993_all = get_descendants('C20993')
print(f'C20993 total descendants (transitive): {len(c20993_all):,}')

# Depth distribution
depth_counts = Counter()
for c in c20993_all:
    depth_counts[get_depth(c, 'C20993')] += 1

print(f'\nDepth distribution:')
for d in sorted(depth_counts):
    print(f'  Depth {d}: {depth_counts[d]:,}')


In [ ]:
# Classify depth-1 children: leaf instruments vs type categories
depth_1 = {c for c in c20993_all if parent_map[c] == 'C20993'}

type_nodes = []  # (code, child_count, name)
leaf_nodes = []  # (code, name)

for c in depth_1:
    cc = len(children_of[c] & c20993_all)  # children that are also C20993 descendants
    if cc > 0:
        type_nodes.append((c, cc, name_map[c]))
    else:
        leaf_nodes.append((c, name_map[c]))

type_nodes.sort(key=lambda x: -x[1])

print(f'Depth-1 leaf instruments: {len(leaf_nodes)}')
print(f'Depth-1 type categories:  {len(type_nodes)}')
print(f'\nType categories (sorted by child count):')
for code, count, name in type_nodes:
    print(f'  {code:10s} {count:5d} children  {name}')


## 3. C211913 — CDISC QRS Instruments Questions

The question-container layer. 365 direct children, each grouping
the individual question concepts that correspond to SDTM CT TESTCDs.


In [ ]:
c211913_children = {c for c in parent_map if parent_map[c] == 'C211913'}
print(f'C211913 direct children: {len(c211913_children)}')

# Each child should contain question concepts as descendants
sub_question_counts = {}
for sub in c211913_children:
    desc = get_descendants(sub)
    sub_question_counts[sub] = len(desc)

total_q = sum(sub_question_counts.values())
print(f'Total question concepts under C211913 children: {total_q:,}')
print(f'Range: {min(sub_question_counts.values())} to {max(sub_question_counts.values())} per container')


## 4. Structural mapping: C211913 question containers → green codelists

For each C211913 child, walk its NCIt descendants and check which ones
appear as TESTCD NCIt codes in the green instrument extract.
If descendants of a C211913 child all belong to one codelist, we have a
structural 1:1 mapping — no name matching needed.


In [ ]:
# Load green extract: NCIt_code -> codelist_testcd
green_df = pd.read_csv(GREEN_INSTRUMENT_PATH)
print(f'Green extract: {len(green_df):,} rows, {green_df["codelist_testcd"].nunique()} distinct codelists')

ncit_to_codelist = dict(zip(green_df['NCIt_code'], green_df['codelist_testcd']))
green_ncit_codes = set(ncit_to_codelist.keys())

# Map each C211913 child to codelists via its question descendants
container_to_codelist = {}  # C211913_child_code -> set of codelist_testcd
container_question_overlap = {}  # C211913_child_code -> count of overlapping questions

for sub in c211913_children:
    descendants = get_descendants(sub)
    overlap = descendants & green_ncit_codes
    if overlap:
        cls = set(ncit_to_codelist[q] for q in overlap)
        container_to_codelist[sub] = cls
        container_question_overlap[sub] = len(overlap)

mapped = len(container_to_codelist)
unmapped = len(c211913_children) - mapped
one_to_one = sum(1 for cls in container_to_codelist.values() if len(cls) == 1)
one_to_many = sum(1 for cls in container_to_codelist.values() if len(cls) > 1)

print(f'\n=== C211913 → Codelist structural mapping ===')
print(f'Mapped:   {mapped} of {len(c211913_children)} ({100*mapped/len(c211913_children):.1f}%)')
print(f'  1:1:    {one_to_one}')
print(f'  1:many: {one_to_many}')
print(f'Unmapped: {unmapped}')

# Build definitive container->codelist dict (1:1 only)
container_codelist_map = {}
for sub, cls in container_to_codelist.items():
    if len(cls) == 1:
        container_codelist_map[sub] = list(cls)[0]

# Show unmapped
unmapped_containers = c211913_children - set(container_to_codelist.keys())
if unmapped_containers:
    print(f'\nUnmapped C211913 children:')
    for c in sorted(unmapped_containers, key=lambda x: name_map.get(x, '')):
        print(f'  {c:10s} {name_map[c]} ({sub_question_counts[c]} NCIt descendants)')


## 5. Instrument identity mapping: codelists → C20993

This is the value-add — resolving each codelist to its instrument as a biomedical concept.

Three matching strategies, in order of confidence:
1. **Exact match** — codelist instrument_label matches C20993 preferred term or synonym
2. **Suffix-strip** — remove trailing type suffix (Functional Test, Questionnaire, etc.)
3. **Normalized match** — strip common noise words, normalize punctuation, handle abbreviation patterns

Strategy 3 is where we go beyond the initial 69% match rate.


In [ ]:
import re

# Build name lookup for ALL C20993 descendants
# Key: normalized name, Value: (code, original_name, depth)
c20993_name_index = {}

for code in c20993_all:
    depth = get_depth(code, 'C20993')
    n = name_map.get(code, '')
    if n:
        c20993_name_index[n.lower()] = (code, n, depth)
    for syn in syn_map.get(code, []):
        if syn:
            c20993_name_index[syn.lower()] = (code, n, depth)

print(f'C20993 name index: {len(c20993_name_index):,} entries')

# Type suffixes to strip from codelist labels
TYPE_SUFFIXES = [
    ' functional test', ' questionnaire', ' clinical classification',
    ' clinical score', ' score', ' scale', ' grade',
    ' clinical category', ' subscale', ' memory aid',
]

def strip_type_suffix(label):
    """Remove trailing SDTM domain type suffix."""
    lower = label.lower()
    for suffix in TYPE_SUFFIXES:
        if lower.endswith(suffix):
            return label[:len(label)-len(suffix)].strip()
    return label

def normalize_name(name):
    """Normalize for fuzzy matching: lowercase, strip noise, normalize punctuation."""
    n = name.lower()
    # Strip leading 'the '
    if n.startswith('the '):
        n = n[4:]
    # Normalize whitespace around hyphens and dashes
    n = re.sub(r'\s*[-–—]\s*', '-', n)
    # Normalize 'test' vs 'functional test'
    n = n.replace(' test', ' functional test')
    # Remove version markers: V1.0, Version 4, 30JUN2015, etc.
    n = re.sub(r'\s+v\d+\.?\d*', '', n)
    n = re.sub(r'\s+version\s+\S+', '', n)
    n = re.sub(r'\s+\d{1,2}[a-z]{3}\d{4}', '', n)
    # Collapse multiple spaces
    n = re.sub(r'\s+', ' ', n).strip()
    return n

# Build normalized index
c20993_norm_index = {}
for code in c20993_all:
    depth = get_depth(code, 'C20993')
    n = name_map.get(code, '')
    if n:
        c20993_norm_index[normalize_name(n)] = (code, n, depth)
    for syn in syn_map.get(code, []):
        if syn:
            c20993_norm_index[normalize_name(syn)] = (code, n, depth)

print(f'Normalized index: {len(c20993_norm_index):,} entries')


In [ ]:
# Get distinct codelists from green extract
codelist_labels = green_df.drop_duplicates('codelist_testcd')[['codelist_testcd', 'instrument_label']]
codelist_labels = dict(zip(codelist_labels['codelist_testcd'], codelist_labels['instrument_label']))
print(f'Distinct codelists to match: {len(codelist_labels)}')

# Run matching with three strategies
results = []

for cl_tc, inst_label in codelist_labels.items():
    label_lower = inst_label.lower()
    stripped = strip_type_suffix(inst_label)
    stripped_lower = stripped.lower()
    norm_label = normalize_name(stripped)

    # Strategy 1: exact match
    if label_lower in c20993_name_index:
        code, orig, depth = c20993_name_index[label_lower]
        results.append({
            'codelist_testcd': cl_tc, 'instrument_label': inst_label,
            'instrument_ncit_code': code, 'instrument_ncit_name': orig,
            'match_method': 'exact', 'instrument_depth': depth
        })
        continue

    # Strategy 2: suffix-strip match
    if stripped_lower in c20993_name_index:
        code, orig, depth = c20993_name_index[stripped_lower]
        results.append({
            'codelist_testcd': cl_tc, 'instrument_label': inst_label,
            'instrument_ncit_code': code, 'instrument_ncit_name': orig,
            'match_method': 'suffix-strip', 'instrument_depth': depth
        })
        continue

    # Strategy 3: normalized match
    if norm_label in c20993_norm_index:
        code, orig, depth = c20993_norm_index[norm_label]
        results.append({
            'codelist_testcd': cl_tc, 'instrument_label': inst_label,
            'instrument_ncit_code': code, 'instrument_ncit_name': orig,
            'match_method': 'normalized', 'instrument_depth': depth
        })
        continue

    # Strategy 3b: normalized match on suffix-stripped + further normalization
    # Try removing item-count qualifiers: '- 17 Item', '- 24 Item' etc.
    norm2 = re.sub(r'-\s*\d+\s*items?', '', norm_label).strip()
    norm2 = re.sub(r'\s+\d+\s*items?', '', norm2).strip()
    if norm2 != norm_label and norm2 in c20993_norm_index:
        code, orig, depth = c20993_norm_index[norm2]
        results.append({
            'codelist_testcd': cl_tc, 'instrument_label': inst_label,
            'instrument_ncit_code': code, 'instrument_ncit_name': orig,
            'match_method': 'normalized-item-strip', 'instrument_depth': depth
        })
        continue

    # No match
    results.append({
        'codelist_testcd': cl_tc, 'instrument_label': inst_label,
        'instrument_ncit_code': None, 'instrument_ncit_name': None,
        'match_method': 'UNMATCHED', 'instrument_depth': None
    })

match_df = pd.DataFrame(results)

# Summary
print('\n=== Codelist → C20993 Instrument match results ===')
method_counts = match_df['match_method'].value_counts()
for method, count in method_counts.items():
    print(f'  {method:25s} {count:4d}')

total_matched = len(match_df[match_df['match_method'] != 'UNMATCHED'])
total = len(match_df)
print(f'\nTotal matched: {total_matched} of {total} ({100*total_matched/total:.1f}%)')
print(f'Unmatched:     {total - total_matched}')


## 6. Unmatched codelists — pattern analysis

Classify unmatched by the reason they fail: versioned variant,
respondent/format variant, disease-specific module, or genuinely absent from C20993.


In [ ]:
unmatched_df = match_df[match_df['match_method'] == 'UNMATCHED'].copy()
print(f'Unmatched codelists: {len(unmatched_df)}')

def classify_unmatched(label):
    lower = label.lower()
    if any(x in lower for x in ['version', 'v1.', 'v2.', 'v3.', 'v4.',
                                  'edition', '2016', '2022', '2024',
                                  '30jun2015', '27mar2024', '7th edition',
                                  'ninds version']):
        return 'versioned'
    if any(x in lower for x in ['self-report', 'self-administered',
                                  'interviewer', 'proxy', 'parent report',
                                  'caregiver', 'female', 'male',
                                  'clinician version', 'young adult',
                                  'teen report', 'child report',
                                  'toddler', 'copd patients']):
        return 'respondent-variant'
    if any(x in lower for x in ['module', 'palliative', 'colorectal',
                                  'lung cancer', 'myeloma', 'oesophageal',
                                  'leukaemia', 'anal cancer',
                                  'neuromuscular']):
        return 'disease-module'
    return 'other'

unmatched_df['category'] = unmatched_df['instrument_label'].apply(classify_unmatched)

print('\nUnmatched by category:')
for cat, count in unmatched_df['category'].value_counts().items():
    print(f'  {cat:25s} {count}')

print('\n--- "Other" unmatched (instruments potentially missing from C20993) ---')
other_unmatched = unmatched_df[unmatched_df['category'] == 'other']
for _, row in other_unmatched.iterrows():
    print(f"  {row['codelist_testcd']:12s}  {row['instrument_label']}")


## 7. Cross-check: C211913 structural mapping meets C20993 name matching

For every codelist that has BOTH a C211913 container (structural, 97%)
and a C20993 instrument match (name-based), do they tell a consistent story?

This is the quality gate — structural verification of heuristic matching.


In [ ]:
# Reverse the container_codelist_map: codelist -> container_code
codelist_to_container = {v: k for k, v in container_codelist_map.items()}

# Join: for each codelist, what do we have?
combined = []
for cl_tc in codelist_labels:
    row = {'codelist_testcd': cl_tc, 'instrument_label': codelist_labels[cl_tc]}

    # C211913 structural mapping
    container_code = codelist_to_container.get(cl_tc)
    row['container_ncit_code'] = container_code
    row['container_name'] = name_map.get(container_code, '') if container_code else ''

    # C20993 instrument match
    match_row = match_df[match_df['codelist_testcd'] == cl_tc].iloc[0]
    row['instrument_ncit_code'] = match_row['instrument_ncit_code']
    row['instrument_ncit_name'] = match_row['instrument_ncit_name']
    row['match_method'] = match_row['match_method']

    # Coverage flags
    row['has_container'] = container_code is not None
    row['has_instrument'] = pd.notna(match_row['instrument_ncit_code'])

    combined.append(row)

combined_df = pd.DataFrame(combined)

# Coverage matrix
both = len(combined_df[combined_df['has_container'] & combined_df['has_instrument']])
container_only = len(combined_df[combined_df['has_container'] & ~combined_df['has_instrument']])
instrument_only = len(combined_df[~combined_df['has_container'] & combined_df['has_instrument']])
neither = len(combined_df[~combined_df['has_container'] & ~combined_df['has_instrument']])

print('=== Coverage matrix ===')
print(f'  Both container + instrument: {both}')
print(f'  Container only (no instrument match): {container_only}')
print(f'  Instrument only (no container): {instrument_only}')
print(f'  Neither: {neither}')
print(f'  Total: {len(combined_df)}')


## 8. Full mapping table

One row per codelist with all identity columns.
This is the candidate enrichment for the green extract.


In [ ]:
# Build definitive instrument_ncit_code including definition
combined_df['instrument_definition'] = combined_df['instrument_ncit_code'].apply(
    lambda x: defn_map.get(x, '') if pd.notna(x) else ''
)

# Question count per codelist
q_counts = green_df.groupby('codelist_testcd').size().rename('question_count')
combined_df = combined_df.merge(q_counts, left_on='codelist_testcd', right_index=True, how='left')

# Sort by match quality
method_order = {'exact': 0, 'suffix-strip': 1, 'normalized': 2, 'normalized-item-strip': 3, 'UNMATCHED': 9}
combined_df['_sort'] = combined_df['match_method'].map(method_order)
combined_df = combined_df.sort_values(['_sort', 'codelist_testcd']).drop(columns='_sort')

display_cols = ['codelist_testcd', 'instrument_label', 'question_count',
                'instrument_ncit_code', 'instrument_ncit_name', 'match_method',
                'container_ncit_code', 'container_name']

print(f'Full mapping: {len(combined_df)} codelists')
print(f'\nMatched instrument identity: {both + instrument_only} ({100*(both+instrument_only)/len(combined_df):.1f}%)')
print(f'Structural container:        {both + container_only} ({100*(both+container_only)/len(combined_df):.1f}%)')

combined_df[display_cols].head(20)


In [ ]:
print('=== Codelists without instrument identity ===')
no_instrument = combined_df[~combined_df['has_instrument']][display_cols]
print(f'{len(no_instrument)} codelists\n')
for _, row in no_instrument.iterrows():
    print(f"  {row['codelist_testcd']:12s} ({row['question_count']:3.0f} Qs)  {row['instrument_label']}")


## 9. Save interim outputs

Write the mapping table and hierarchy data for further analysis.


In [ ]:
# Save full mapping
out_path = os.path.join(INTERIM_PATH, 'CoA_Instrument_Mapping_Probe.csv')
combined_df[display_cols + ['has_container', 'has_instrument', 'instrument_definition']].to_csv(
    out_path, index=False
)
print(f'Saved: {out_path}')

# Save C20993 hierarchy (all descendants)
c20993_rows = []
for code in c20993_all:
    c20993_rows.append({
        'ncit_code': code,
        'preferred_term': name_map.get(code, ''),
        'parent_code': parent_map.get(code, ''),
        'parent_name': name_map.get(parent_map.get(code, ''), ''),
        'depth': get_depth(code, 'C20993'),
        'synonyms': '|'.join(syn_map.get(code, [])),
        'definition': defn_map.get(code, ''),
    })

c20993_df = pd.DataFrame(c20993_rows).sort_values(['depth', 'preferred_term'])
c20993_path = os.path.join(INTERIM_PATH, 'C20993_Instrument_Hierarchy.csv')
c20993_df.to_csv(c20993_path, index=False)
print(f'Saved: {c20993_path} ({len(c20993_df):,} rows)')

# Save C211913 children with codelist mapping
c211913_rows = []
for code in c211913_children:
    cl = container_codelist_map.get(code)
    c211913_rows.append({
        'ncit_code': code,
        'preferred_term': name_map.get(code, ''),
        'codelist_testcd': cl if cl else '',
        'question_count': sub_question_counts.get(code, 0),
        'mapped_structurally': cl is not None,
    })

c211913_df = pd.DataFrame(c211913_rows).sort_values('preferred_term')
c211913_path = os.path.join(INTERIM_PATH, 'C211913_Question_Containers.csv')
c211913_df.to_csv(c211913_path, index=False)
print(f'Saved: {c211913_path} ({len(c211913_df)} rows)')


## 10. Probe findings

### C20993 is a type taxonomy, not a flat instrument list
- 273 depth-1 children: 243 leaf instruments + 30 type categories
- Type categories contain 1,000+ instruments at depth 2-3
- Total leaf-level instruments: ~2,000
- The plan's '274 instruments' was depth-1 only — misses the actual instruments at depth 2+

### C211913 → Codelist: structural, reliable
- 97% of C211913 children map 1:1 to a green codelist via NCIt parent-child walks
- No name matching needed — purely structural
- This is the verified backbone

### Codelist → C20993 Instrument: name-based, improvable
- Exact match: ~57%
- With suffix-strip: ~69%
- With normalization: check results above
- Remaining unmatched: versioned variants, respondent forms, disease modules, and
  some genuinely absent instruments (Trail Making Test, Fatigue Severity Scale, etc.)

### Practical implication
- Each codelist can be enriched with an `Instrument_NCIt_Code` (C20993 descendant) at 70-85% confidence
- The C211913 container provides the structural anchor for verification
- Unresolved codelists are explicitly flagged, not silently dropped
- The value is the instrument identity — the biomedical concept a SoA catalog can match against
